# Pre-train multigauge on CAMELS-US and validate on real basins

1. pre-trains the **LSTM** (probabilistic CMAL output) on the 531 CAMELS-US basins, and optionally the **δHBV hybrid**;
2. validates on **five large U.S. basins the model never saw**, with sub-basin and forecast tests for the Washita.

**Setup:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. Approve the Google Drive prompt.

**Interruptions are safe.** Everything is saved to `My Drive/multigauge/` as it goes: the CAMELS download, training
checkpoints after every epoch, finished models and each basin's results. If the session drops, reconnect and
*Run all* again; finished steps are skipped and training resumes where it stopped.

Data: CAMELS v1.2 (Newman et al., 2015; Addor et al., 2017; NCAR GDEX, Zenodo record 15529996); USGS; Daymet; NOAA GHCN-Daily.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
WORK = Path("/content/drive/MyDrive/multigauge"); WORK.mkdir(parents=True, exist_ok=True)
print("Saving everything to", WORK)

## 1. Install multigauge

In [ ]:
!rm -rf multigauge-hydro && git clone -q https://github.com/ogunbajoadeyinka/multigauge-hydro.git
!pip install -q -e "multigauge-hydro[download]"

## 2. CAMELS-US (kept on Drive after the first download)

In [ ]:
import subprocess, os
ZIP = WORK / "camels_v1p2_metForcing_obsFlow.zip"
TOPO = WORK / "camels_topo.txt"
BASE = "https://zenodo.org/api/records/15529996/files"
if not ZIP.exists() or ZIP.stat().st_size < 3_400_000_000:
    subprocess.run(["wget", "-nv", "-O", str(ZIP) + ".part", f"{BASE}/basin_timeseries_v1p2_metForcing_obsFlow.zip/content"], check=True)
    os.replace(str(ZIP) + ".part", ZIP)
if not TOPO.exists():
    subprocess.run(["wget", "-nv", "-O", str(TOPO), f"{BASE}/camels_topo.txt/content"], check=True)
CAMELS = Path("/content/camels"); CAMELS.mkdir(exist_ok=True)
if not (CAMELS / "basin_dataset_public_v1p2").exists():
    subprocess.run(["unzip", "-q", "-o", str(ZIP), "-d", str(CAMELS)], check=True)
subprocess.run(["cp", str(TOPO), str(CAMELS)], check=True)
print(sorted(p.name for p in CAMELS.iterdir()))

## 3. Load the 531 basins

In [ ]:
import pickle, numpy as np, pandas as pd, multigauge as mg
from multigauge.camels import load_camels
CACHE = WORK / "camels_531_basins.pkl"
if CACHE.exists():
    basins = pickle.load(open(CACHE, "rb"))
else:
    ids = open("multigauge-hydro/notebooks/camels_531_basins.txt").read().split()
    basins = load_camels(CAMELS, ids, progress=lambda i, n, b: print(f"\r{i}/{n}", end=""))
    pickle.dump(basins, open(CACHE, "wb"))
print("\nloaded", len(basins), "basins")

## 4. Pre-train the LSTM (about 30–60 min; resumes after interruptions)

Standard CAMELS periods (Kratzert et al., 2019): train 1980-10-01 to 1995-09-30, validate 1995-10-01 to 2000-09-30, test 2000-10-01 to 2010-09-30.

In [ ]:
from multigauge.nn import ModelConfig, StreamflowModel
from multigauge.transfer import pretrain
MODEL_DIR = WORK / "pretrained" / "camels-us"
if (MODEL_DIR / "model.json").exists():
    model = StreamflowModel.load(MODEL_DIR)
    print("Already trained:", model.info.get("best_val_score"))
else:
    cfg = ModelConfig.preset("pretrain")
    print(cfg)
    model = pretrain(basins, cfg, ("1980-10-01", "1995-09-30"), ("1995-10-01", "2000-09-30"),
                     checkpoint=WORK / "pretrain_checkpoint.pt",
                     progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
    model.info.update(source="CAMELS-US", description="LSTM pre-trained on 531 CAMELS-US basins (Daymet forcings, CMAL output).")
    model.save(MODEL_DIR)
!mkdir -p multigauge-hydro/src/multigauge/pretrained && cp -r "{MODEL_DIR}" multigauge-hydro/src/multigauge/pretrained/

## 5. Test-period skill on the 531 basins (2000-10-01 to 2010-09-30)

In [ ]:
from multigauge.transfer import period_mask
TEST_FILE = MODEL_DIR / "test_metrics.csv"
if TEST_FILE.exists():
    test = pd.read_csv(TEST_FILE, index_col=0)
else:
    model.net.to("cuda" if torch.cuda.is_available() else "cpu")
    rows = []
    for b in basins:
        m = period_mask(b.dates, ("2000-10-01", "2010-09-30"))
        rows.append({"basin": b.basin_id, **mg.evaluate_probabilistic(b.q_mm.to_numpy()[m], model.predict(b)[m])})
    model.net.to("cpu")
    test = pd.DataFrame(rows).set_index("basin"); test.to_csv(TEST_FILE)
print(test[["NSE", "KGE", "CRPS", "Coverage90"]].median().round(3))

*Reference:* published single-forcing CAMELS LSTMs reach a median test NSE of roughly 0.72–0.74 using many more catchment attributes; this model uses only attributes computed from the weather data so it runs on any U.S. basin.

## 6. Pre-train the δHBV hybrid (about 20–40 min)

In [ ]:
from multigauge.hybrid import HybridConfig, HybridModel
HYB_DIR = WORK / "pretrained" / "camels-us-dhbv"
if (HYB_DIR / "hybrid.json").exists():
    hyb = HybridModel.load(HYB_DIR)
else:
    tr = [period_mask(b.dates, ("1980-10-01", "1995-09-30")) for b in basins]
    va = [period_mask(b.dates, ("1995-10-01", "2000-09-30")) for b in basins]
    hyb = HybridModel(HybridConfig(epochs=20)).fit(basins, tr, va,
            progress=lambda e, n, s: print(f"epoch {e}/{n}: median validation NSE {s:.3f}", flush=True))
    hyb.save(HYB_DIR)

## 7. Validate on five basins the model never saw

All are larger than 2,000 km², so none is among the 531 pre-training basins. For each: USGS boundary and flow,
Daymet weather and the NOAA GHCN-Daily rain gauges inside the basin, then the full gauge-value study.
Periods: train water years 2002–2013, validate 2014–2017, test 2018–2023.

In [ ]:
import traceback
from multigauge.prepare import prepare_usgs_basin
from multigauge.study import StudyConfig, load_saved_study
from multigauge.report import summary_table
SITES = {
    "07326500": "Washita River at Anadarko, OK",
    "05453100": "Iowa River at Marengo, IA",
    "02347500": "Flint River, GA",
    "08167000": "Guadalupe River at Comfort, TX",
    "01643000": "Monocacy River at Jug Bridge, MD",
}
START, END = "2000-10-01", "2023-09-30"
PERIODS = dict(train_period=("2001-10-01", "2013-09-30"), validation_period=("2013-10-01", "2017-09-30"),
               test_period=("2017-10-01", "2023-09-30"))
OUT = WORK / "validation"; OUT.mkdir(exist_ok=True)
for site, label in SITES.items():
    if (OUT / site / "results" / "run_info.json").exists():
        print(f"[{site}] already done"); continue
    try:
        if (OUT / site / "data" / "basin.json").exists():
            d = mg.BasinData.from_folder(OUT / site / "data")
        else:
            d = prepare_usgs_basin(site, START, END, OUT / site / "data", name=label)
        res = mg.run_study(d, model, StudyConfig(n_seeds=3, finetune_epochs=20, **PERIODS), hybrid=hyb,
                           progress=lambda f, m: print(f"\r[{site}] {int(f*100):3d}% {m[:70]:<70}", end=""))
        print()
        out = res.save(OUT / site / "results"); mg.write_report(res, d, out / "report.html")
        print(summary_table(res).round(3)[["NSE", "KGE", "CRPS"]].to_string(), "\n")
    except Exception as exc:
        print(f"\n[{site}] FAILED: {exc}"); traceback.print_exc()

## 8. Washita: sub-basins and forecasts

In [ ]:
import json
from shapely.geometry import mapping
from multigauge.routing import SubbasinSet, auto_zones, run_routed_study
from multigauge.forecast import ForecastConfig, ForecastModel
from multigauge.download import daymet_basin
from multigauge.study import make_periods
site = "07326500"; R = OUT / site / "results"
if (R / "run_info.json").exists() and not (R / "forecast_skill.csv").exists():
    d = mg.BasinData.from_folder(OUT / site / "data")
    zdir = OUT / site / "data" / "subbasins"; zdir.mkdir(exist_ok=True)
    zones = auto_zones(d.boundary, d.outlet, 4); feats = []
    for i, z in enumerate(zones):
        zid = f"Z{i+1}"
        if not (zdir / f"{zid}.csv").exists():
            daymet_basin(z, START, END, n_points=9).to_csv(zdir / f"{zid}.csv", date_format="%Y-%m-%d")
        feats.append({"type": "Feature", "geometry": mapping(z), "properties": {"id": zid, "downstream": None if i == 0 else f"Z{i}"}})
    (zdir / "subbasins.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": feats}))
    periods = make_periods(d.dates, StudyConfig(**PERIODS), model.cfg.lookback)
    routed = run_routed_study(d, model, SubbasinSet.from_folder(zdir), periods, routing="gamma", epochs=15)
    routed.summary.to_csv(R / "subbasin_models.csv"); routed.contributions.to_csv(R / "subbasin_contributions.csv")
    print(routed.summary.xs("test", level="period")[["NSE", "KGE"]].round(3))
    b = d.to_basin(); m = {k: np.asarray((d.dates >= a) & (d.dates <= e)) for k, (a, e) in periods.items()}
    ft = mg.finetune(model, b, m["train"], m["validation"], epochs=20)
    fm = ForecastModel(ft, ForecastConfig(epochs=15)).fit(b, m["train"], m["validation"])
    iss = np.where(m["test"])[0]; iss = iss[iss < len(b.dates) - 8]
    skill = fm.skill_by_lead(b, iss); skill.to_csv(R / "forecast_skill.csv", index=False)
    print(skill.round(3).to_string(index=False))

## 9. Summary across basins

In [ ]:
rows = []
for site in SITES:
    if not (OUT / site / "results" / "run_info.json").exists(): continue
    res = load_saved_study(OUT / site / "results")
    s = res.summary.xs("test", level="period")
    gv = res.gauge_value[res.gauge_value["period"] == "test"].set_index("gauge_input")
    r = {"site": site, "basin": res.basin_name, "area_km2": round(res.area_km2), "gauges": len(res.station_summary)}
    for k in ["gr4j", "local", "pretrained", "finetuned", "ft_voronoi", "ft_disaggregated", "dhbv", "ensemble"]:
        if k in s.index: r[f"NSE_{k}"] = round(s.loc[k, "NSE"], 3)
    best = gv["NSE_gain"].idxmax()
    r.update(best_gauge_input=best, NSE_gain_from_gauges=round(gv.loc[best, "NSE_gain"], 3),
             CRPS_reduction_pct=round(gv.loc[best, "CRPS_reduction_pct"], 1))
    rows.append(r)
summary = pd.DataFrame(rows); summary.to_csv(OUT / "summary.csv", index=False); summary

## 10. Download

Attach `multigauge_results.zip` in your chat with Claude. (Everything is also in `My Drive/multigauge/`.)

In [ ]:
!cd "{WORK}" && rm -f /content/multigauge_results.zip && zip -qr /content/multigauge_results.zip pretrained validation -x "*.pt" && ls -la /content/multigauge_results.zip
from google.colab import files
files.download("/content/multigauge_results.zip")

## Optional: Washita with your Oklahoma Mesonet gauges

Run this cell on its own after the above. Upload `rainfall.csv` (date + one column per station, mm/day) and
`stations.csv` (id, lat, lon). Results stay in your Drive; check the Mesonet terms before sharing them.

In [ ]:
import shutil
up = files.upload()
if {"rainfall.csv", "stations.csv"} <= set(up):
    mdir = OUT / "07326500_mesonet" / "data"; mdir.mkdir(parents=True, exist_ok=True)
    for f in ("forcings.csv", "streamflow.csv", "boundary.geojson", "basin.json"):
        shutil.copy(OUT / "07326500" / "data" / f, mdir / f)
    for f in ("rainfall.csv", "stations.csv"):
        (mdir / f).write_bytes(up[f])
    dm = mg.BasinData.from_folder(mdir, name="Washita River at Anadarko, OK (Mesonet gauges)")
    rm = mg.run_study(dm, model, StudyConfig(n_seeds=3, finetune_epochs=20, run_ablation=True, **PERIODS), hybrid=hyb)
    out = rm.save(OUT / "07326500_mesonet" / "results"); mg.write_report(rm, dm, out / "report.html")
    print(summary_table(rm).round(3)[["NSE", "KGE", "CRPS"]].to_string())